<a href="https://colab.research.google.com/github/SurajTamang-22/CodeAlpha_MUSIC-PLAYER/blob/main/Adhar_extract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y tesseract-ocr
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (300 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!pip install -q pytesseract opencv-python langchain_ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh|sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nohup ollama serve &
%env OLLAMA_HOST=0.0.0.0

nohup: appending output to 'nohup.out'
env: OLLAMA_HOST=0.0.0.0


In [ ]:
!ollama pull mistral:7b

In [ ]:
import gradio as gr
import pytesseract
import cv2
import json
import tempfile
from langchain_ollama.llms import OllamaLLM

In [ ]:
model = OllamaLLM(model="mistral:7b",format="json")

In [ ]:
class AadhaarInfoExt:

    def __init__(self, job_id, image_path):
        self.job_id = job_id
        self.image_path = image_path

    def image_load(self):
        self.img = cv2.imread(self.image_path)
        if self.img is None:
            raise ValueError("Image not found or unreadable")

    def image_processing(self):
        self.preprocessed_img = cv2.cvtColor(self.img, cv2.COLOR_BGR2GRAY)

    def acr(self):
        custom_config = r'--oem 3 --psm 6' # Tesseract configuration for OCR
        self.ocr_text = pytesseract.image_to_string(
            self.preprocessed_img,
            config=custom_config
        )

    def extract(self):

        self.image_load()
        self.image_processing()
        self.acr()

        prompt = f"""

You are a structured document extraction assistant.

Your task is to extract the following fields from the OCR-scanned text of an Aadhaar card:
- Username (Full Name)
- Address
- Phone (Mobile Number)
- Aadhaar Number

Extraction Rules:
- Username should be the full name as it appears on the card.
- Address should be the complete mailing address.
- Phone should be a 10-digit numeric string.
- Aadhaar Number should be a 12-digit numeric string, usually found in groups of four digits (e.g., XXXX XXXX XXXX).
- plsease check thoroughly and give good result in hazzy photos too.

Important Instructions:
- Only return valid JSON
- Do not explain
- Do not wrap in markdown

Format:
{{
"Username": "...",
"Address": "...",
"Phone": "...",
"Aadhaar Number": "..."
}}

OCR Text:
{self.ocr_text}

"""

        response = model.invoke(prompt)
        result = json.loads(response)

        return result

def extract_aadhaar_info(image):

    if image is None:
        return "Please upload an image"

    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp_file:

        image.save(tmp_file.name)

        extractor = AadhaarInfoExt(
            job_id="1",
            image_path=tmp_file.name
        )

        try:
            result = extractor.extract()

            plain_text = (
                f"Username: {result.get('Username','N/A')}\n"
                f"Address: {result.get('Address','N/A')}\n"
                f"Phone: {result.get('Phone','N/A')}\n"
                f"Aadhaar Number: {result.get('Aadhaar Number','N/A')}"
            )

            return plain_text

        except Exception as e:
            return f"Error: {e}"

with gr.Blocks() as demo:

    gr.Markdown("## Aadhaar Card Info Extractor")

    with gr.Row():

        with gr.Column():
            image_input = gr.Image(type="pil", label="Upload Aadhaar Card Image")
            clear_btn = gr.Button("Clear")

        with gr.Column():
            result_output = gr.Textbox(label="Extracted Info", lines=10)
            submit_btn = gr.Button("Extract Info")

    submit_btn.click(
        fn=extract_aadhaar_info,
        inputs=image_input,
        outputs=result_output
    )

    clear_btn.click(
        fn=lambda: (None, ""),
        inputs=[],
        outputs=[image_input, result_output]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://585d4c188813288da7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
